# Experiment 1 — XGBoost Baseline (No Imbalance Handling)

Standard XGBoost with zero modification. Worst-case reference point.
Shows how badly standard training fails on imbalanced data.

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# FIXED HYPERPARAMETERS — DO NOT CHANGE ACROSS ANY XGB EXPERIMENT
PARAMS = dict(n_estimators=500, max_depth=6, learning_rate=0.1,
              random_state=42, eval_metric='auc', verbosity=0,
              use_label_encoder=False)
print("Experiment 1 — XGBoost Baseline")
print("Params:", PARAMS)

Experiment 1 — XGBoost Baseline
Params: {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.1, 'random_state': 42, 'eval_metric': 'auc', 'verbosity': 0, 'use_label_encoder': False}


In [2]:
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp1-XGB] Training Version {v}...")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train = train.drop('label',axis=1).values; y_train = train['label'].values
    X_test  = test.drop('label',axis=1).values;  y_test  = test['label'].values

    model = XGBClassifier(**PARAMS)
    t0    = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    probs = model.predict_proba(X_test)[:,1]
    preds = model.predict(X_test)

    metrics = compute_all_metrics(y_test, probs, preds, train_time)
    metrics['Version'] = v
    all_results.append(metrics)

    np.save(os.path.join(RESULTS_DIR, f'probs_exp1_xgb_version_{v}.npy'), probs)
    np.save(os.path.join(RESULTS_DIR, f'labels_version_{v}.npy'), y_test)
    print_metrics(metrics, label=f"Exp1-XGB Version {v}")

print("\n[Exp1-XGB] Complete.")


[Exp1-XGB] Training Version A...
[Exp1-XGB Version A] AUC=0.8219 | F1=0.7572 | Signal_Sig=178.6683 | Train_Time=20.76s

[Exp1-XGB] Training Version B...
[Exp1-XGB Version B] AUC=0.8115 | F1=0.2300 | Signal_Sig=26.2529 | Train_Time=405.88s

[Exp1-XGB] Training Version C...
[Exp1-XGB Version C] AUC=0.7737 | F1=0.0188 | Signal_Sig=3.6909 | Train_Time=570.82s

[Exp1-XGB] Complete.


In [3]:
results_df = pd.DataFrame(all_results)[['Version','AUC','F1','Signal_Significance','Train_Time_sec']]
save_path  = os.path.join(RESULTS_DIR, 'experiment1_baseline_xgb.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      AUC       F1  Signal_Significance  Train_Time_sec
      A 0.821946 0.757176           178.668317           20.76
      B 0.811544 0.230017            26.252928          405.88
      C 0.773673 0.018848             3.690886          570.82

✓ Saved → ..\results\experiment1_baseline_xgb.csv
